# Lesson 11 · Three rulers for one vector — vector norms (L1 / L2 / ∞) and the geometry of normalization

**Goal**: compute L1/L2/∞ norms and distances, and normalize a vector to unit length.

> Before you write `normalize`, read the "Digging deeper" on why you can't normalize by sum or max. The details of Lasso regularization can wait for the training lessons.

**Why it matters for Aurora**: MSE ≈ the squared L2 norm of the error; MFCC features are often L2-normalized before they go into a model, so that frames with different energy sit on the same scale.

← **Previous lesson** · [L10 · Dot products and projection](L10_dot_product.ipynb)

> Last lesson covered **dot products and projection**: a·b = |a||b|cosθ, and why similarity = dot product ÷ norms.
> This lesson explores **vector norms**.

## This lesson's story: there's more than one ruler for "distance"

When you measure the distance between two points on a map, there are several ways to do it:
- **Straight-line distance** (the way a bird flies): use the Pythagorean theorem — this is the **L2 norm**
- **City-block distance** (the way a taxi drives along the streets): go along x, then along y — this is the **L1 norm**
- **The longest single stretch** (how a king moves on a chessboard): look only at the largest component — this is the **L∞ norm**

Three "rulers," three different definitions of "length." Which one fits depends on the task:
- **L2**: geometric distance, MFCC normalization, weight decay in neural networks (L2 regularization)
- **L1**: sparsity, Lasso regression, more robust to outliers
- **L∞**: worst-case error bounds

This lesson's only exercise: implement `normalize(v)` — squeeze any vector onto the unit L2 sphere (so its length becomes 1).

## 1. Three norms, three rulers

Given a vector `v = [v₁, v₂, ..., vₙ]`, here are three definitions of "length":

| Norm | Formula | Geometric meaning | Use in Aurora |
|---|---|---|---|
| L2 (Euclidean) | `√(Σ vᵢ²)` | straight-line distance (the Pythagorean theorem, generalized) | MFCC normalization, the denominator of cosine similarity |
| L1 (Manhattan) | `Σ |vᵢ|` | city-block distance | Lasso regularization, robust to outliers |
| L∞ (Chebyshev) | `max(|vᵢ|)` | the largest component | worst-case error bound |

**Practice by hand** (v = [3, 4]; work it out on paper first, then run the code to check):

| Norm | By hand |
|---|---|
| L2 | ✏️ √(9+16) = ? |
| L1 | ✏️ 3+4 = ? |
| L∞ | ✏️ max(3,4) = ? |

### Aside: who is "Chebyshev," and what does L∞ actually measure?

In the table, L∞ has "Chebyshev" in brackets after it — that's the name of the mathematician **Pafnuty Chebyshev**. He studied questions like "how big is the error in the worst case?", and the L∞ norm (which looks only at the largest component) is exactly the tool for measuring "how big is the worst single deviation," so the measure was later named after him. It also has a more down-to-earth nickname — **chessboard distance / Chebyshev distance** — because in chess the king can move one square horizontally, vertically or diagonally, and each of those counts as "one step." The number of steps it needs to reach a target square equals **the larger** of the horizontal and vertical distances, which is precisely the definition of L∞.

If the chessboard picture still feels unfamiliar, try an everyday scene instead: a quality inspector checks a shirt, measuring how far the chest, sleeve length and waist are from the standard size: `[+2cm, -1cm, +5cm]`. The inspector usually doesn't care how big the three deviations are added together (that's L1 thinking); they only care about **the one that's furthest out of spec** — `|+5cm|` is the largest, so the shirt is judged as "off by 5cm" when deciding whether it passes. That's L∞: **ignore the total, look only at the worst single component** — which makes it a natural fit for "worst-case error bound" situations.



### Notation Q&A: how is `‖v‖` related to the `|x|` you learned at school?

At school you learned `|x|` (absolute value), e.g. `|-5|=5` — it's a ruler for measuring how far **a single number** is from 0, and the rule is simple: drop the minus sign.

But the vector `v=[3,4]` isn't one number; it's several numbers bundled together. "How far from the origin" can't just mean dropping minus signs — you need the Pythagorean theorem, or some other formula, to combine the components into one length. To remind readers that "this isn't an ordinary absolute value, but a more elaborate length defined specially for vectors (and later for matrices)," mathematicians simply invented a new symbol — double bars `‖ ‖` — used just for vectors and matrices.

In short:
- `|x|`: for **a number**; one fixed formula (drop the minus sign).
- `‖v‖`: for **a vector**; there's more than one formula (Pythagoras, sum of absolute values, …), so you write a subscript at the bottom right, like `‖v‖₂` or `‖v‖₁`, to say which one you mean. You'll see what each subscript stands for in a moment.



## Notation first: norms first, then normalization

The L2 norm is written `‖v‖₂`, and L1 is written `‖v‖₁`. For the vector `v = [3, 4]`: `‖v‖₂ = sqrt(9+16) = 5`, `‖v‖₁ = 3+4 = 7`. Both norms measure "length," but L2 punishes large components more heavily, while L1 treats every component the same.

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np
v = np.array([3.0, 4.0])
print('by hand:', np.sqrt(np.sum(v**2)))   # 5.0
print('numpy:', np.linalg.norm(v))      # 5.0
print('L1 norm (sum of absolute values):', np.sum(np.abs(v)))

## Hands-on: one vector, two lengths

Run the code below and notice that L1 and L2 give different values, yet after normalization the vector's L2 norm is exactly 1. Change the size of one component and watch which norm changes more sharply.

In [ ]:
import numpy as np

# Numerical comparison of the L1, L2 and L∞ norms
vecs = [np.array([3., 4.]), np.array([1., 0., 0., 10.]), np.array([-2., 2., -2.])]
for v in vecs:
    l1 = np.sum(np.abs(v))
    l2 = np.sqrt(np.sum(v**2))
    linf = np.max(np.abs(v))
    print(f'v={v}  L1={l1:.2f}  L2={l2:.2f}  L∞={linf:.2f}')
print('Notice: L2 is smaller than L1 (triangle inequality), and L∞ is the smallest (it only looks at the largest component).')


### Digging deeper: why does L∞ ≤ L2 ≤ L1 always hold?

First, let's clear up a common misunderstanding: **these three norms are not competing over "who gives the shorter, truer distance."** They are three completely different rulers, each serving a different task; none of them is right or wrong. Take `v=[1, 100]`: when L∞ says the "distance" is 100, it isn't saying "going along the diagonal only takes 100"; it's saying "if all you care about is the largest component of this vector, it's 100" — that's simply the definition of L∞, and it has nothing to do with shortest paths. Which ruler you pick depends on whether you care about "the worst single item" (L∞), "the total deviation" (L1), or "geometric straight-line distance" (L2).

Still, for **the same** vector, the numbers the three rulers give really do always satisfy `L∞ ≤ L2 ≤ L1`. That's no coincidence — it follows directly from the formulas (let `v=[v₁,...,vₙ]`, and write `M = max(|v₁|,...,|vₙ|)` for the absolute value of the largest component, i.e. L∞):

**Step 1, L∞ ≤ L2:**
```
L2² = v₁² + v₂² + ... + vₙ²  ≥ M²   (because M is one of the |vᵢ|, and the other terms are all ≥ 0, so they can only make the sum bigger)
so L2 = √(L2²) ≥ √(M²) = M = L∞
```

**Step 2, L2 ≤ L1:**
```
L1² = (|v₁| + |v₂| + ... + |vₙ|)²
    = v₁² + v₂² + ... + vₙ²  +  2·(sum of all cross terms |vᵢ|·|vⱼ|, i≠j)
    ≥ v₁² + v₂² + ... + vₙ²   (the cross terms are all ≥ 0, so dropping them can only make the expression smaller)
    = L2²
so L1 = √(L1²) ≥ √(L2²) = L2
```

Put the two steps together: `L∞ ≤ L2 ≤ L1` holds for every vector in every dimension — no need to check examples one by one. Below we get a feel for it geometrically: the three kinds of "unit ball" (the shape formed by all points with `‖v‖=1`) nest inside one another, and whichever ruler "reaches 1 with smaller coordinates" has the bigger ball.



In [ ]:
from laviz import style
import matplotlib.pyplot as plt
import numpy as np

style()
fig, ax = plt.subplots(figsize=(5, 5))

# L∞ unit ball: a square with corners at (±1, ±1)
square = np.array([[1, 1], [-1, 1], [-1, -1], [1, -1], [1, 1]])
ax.plot(square[:, 0], square[:, 1], color='#E76F51', lw=2, label='L∞ ball (square, largest)')

# L2 unit ball: the unit circle
theta = np.linspace(0, 2 * np.pi, 200)
ax.plot(np.cos(theta), np.sin(theta), color='#2A9D8F', lw=2, label='L2 ball (circle, middle)')

# L1 unit ball: a diamond with corners at (±1,0) and (0,±1)
diamond = np.array([[1, 0], [0, 1], [-1, 0], [0, -1], [1, 0]])
ax.plot(diamond[:, 0], diamond[:, 1], color='#E9C46A', lw=2, label='L1 ball (diamond, smallest)')

ax.axhline(0, color='k', lw=0.5)
ax.axvline(0, color='k', lw=0.5)
ax.set_aspect('equal')
ax.set_title('Three unit balls (‖v‖=1), nested')
ax.legend(loc='upper right', fontsize=9)
plt.show()

print('The square (L∞) encloses the circle (L2), and the circle encloses the diamond (L1).')
print('A bigger ball means it is "easier" to reach "length = 1" (smaller coordinates suffice), so that norm gives smaller values —')
print('which is exactly what L∞ ≤ L2 ≤ L1 looks like as a picture.')


## Code experiment: loop over a few vectors and compare the L1 / L2 norms

The L2 norm is more sensitive to extreme values among the components — one very large component pushes L2 up dramatically, while its effect on L1 is only linear. The loop below shows how this gap changes with how "spiky" the vector is.

In [ ]:
import numpy as np

# Sensitivity to outliers: L1 is more robust than L2
base = np.ones(10)
for outlier in [0, 1, 5, 10, 50]:
    v = base.copy(); v[0] = outlier
    print(f'v[0]={outlier:>3d}  L1={np.linalg.norm(v,1):.1f}  '
          f'L2={np.linalg.norm(v,2):.2f}  L∞={np.linalg.norm(v,np.inf):.0f}')
print('→ L2 is amplified by the outlier (squaring), L1 grows linearly; once the outlier ≥ the other components, L∞ simply equals the outlier (for v[0]=0 and 1, L∞ is still the other components\' value of 1).')


## 2. The distance between two vectors = the length of their difference

**First get this logic straight**: why is the distance between two points "compute the difference vector, then take its length," rather than some other operation applied directly to the coordinates?

Imagine you're standing at point `A=(1,2)` on a map and your friend is at point `B=(4,6)`. How would you work out how far apart you are? Intuitively you'd say: "We're `4-1=3` apart horizontally and `6-2=4` apart vertically, so by Pythagoras it's `√(3²+4²)=5`."

Translate that intuition into the language of vectors and it's exactly two steps:
1. **First work out "how far right and how far up I have to move to get from A to B"** — that's vector subtraction, `b - a = [4-1, 6-2] = [3, 4]`. The result is itself a vector: it's "the displacement arrow joining A and B," pointing from A to B.
2. **Then measure how long that arrow is** — take the norm of the difference vector, `‖b - a‖₂ = √(3²+4²) = 5`.

So "the distance between two points" isn't some new formula invented out of thin air; it's "find the displacement vector, then measure its length with the norm you already know" — and the coordinates of the difference vector `b-a` are exactly the component-by-component differences. That's why `dist(a, b) = ‖a - b‖`: the norm takes the idea of "how long a vector is" and reuses it to measure "how far apart two points are." The code below checks this 2D example and plots the three vectors a, b and b-a.

In [ ]:
a = np.array([1.0, 2.0]); b = np.array([4.0, 6.0])
print('distance =', np.linalg.norm(a - b))  # 5.0

In [ ]:
from laviz import style, arrows2d
style()
arrows2d([a, b, b - a], ['a=(1,2)', 'b=(4,6)', 'b-a=(3,4), length 5'],
         title='distance = length of the difference vector (b-a)')


## 3. ✏️ Your task: implement `normalize` (scale to length 1)

**Line of reasoning**:
1. The input `v` is a 1-D array of shape `(n,)`; the goal is an output vector whose L2 norm equals 1.
2. First compute `length = np.linalg.norm(v)` — the vector's "total length."
3. Use `v / length` to shrink every component by the same factor: the ratios between components (the direction) stay the same, and the length goes from `length` to `length/length = 1`. Key point: you can't use `v / np.sum(v)` or `v / np.max(v)` — those don't guarantee `‖result‖₂ = 1`.
4. **The zero-vector edge case**: if `length == 0`, `v / length` produces NaN. Convention: the zero vector should return the all-zero vector. Catch it early with `if length == 0: return v` — non-zero vectors take the exact path `v / length` (norm exactly 1), and the zero vector is returned as is, all zeros: no division by zero, and no error introduced.

**Why this convention, rather than just raising an exception?** Think of Aurora's MFCC feature extraction: if a frame of audio happens to be **completely silent** (say the start of a recording, or the few frames where a speaker pauses), the feature vector computed for that frame may be exactly all zeros `[0,0,...,0]`. The zero vector has no direction — it doesn't point anywhere — so "making the zero vector a unit vector" is simply undefined mathematically. You then have two options:
- **Raise an exception** and halt the whole pipeline — but silence is a perfectly normal input (pauses in the middle of a conversation are common), and the program shouldn't fall over because of one frame;
- **Return the all-zero vector** — its length is still 0, and you haven't made up a direction that doesn't exist; downstream code (computing cosine similarity, for example) that receives an all-zero vector naturally gets a similarity of 0 ("not like anything"), instead of letting `0/0` produce `NaN` and contaminate every calculation that depends on it from then on.

Both options "make sense mathematically," but in engineering terms the second is safer: **the convention is never to let NaN appear, and to encode "no direction, undetermined" as "zero" rather than as "the program falls over"** — which is also why many numerical libraries (Aurora's own code included) generally choose to return a safe default rather than raise an exception when there's a risk of dividing by zero.

**Reference input and output**: `v=[3,4]` → norm 5, normalized to `[0.6, 0.8]`; check that `0.6²+0.8²=1.0`

> 💡 Need a hint? Once you've finished the exercise, you can compare with the reference implementation in the `solutions/` folder.


### Digging deeper: why can't you use `v / np.sum(v)` or `v / np.max(v)`?

Intuitively, dividing by "the sum" or "the maximum" also seems to "shrink" the vector. But our goal is very specific: make the result's **L2 norm** (the length you get from Pythagoras) come out at exactly 1 — not just make it smaller somehow. Let's simply run all three divisions and see what L2 norm each result actually has:

In [ ]:
v_demo = np.array([3.0, 4.0])
print('original v:', v_demo)
for name, divisor in [('sum(v)', np.sum(v_demo)), ('max(v)', np.max(v_demo)),
                       ('‖v‖₂ (correct)', np.linalg.norm(v_demo))]:
    result = v_demo / divisor
    print(f'v / {name:11s} = {divisor:.1f}  ->  result={result}  L2 norm of result={np.linalg.norm(result):.4f}')


In the output, only the row that divides by `‖v‖₂` gives a result whose L2 norm is exactly `1.0000`; the vectors you get by dividing by `sum` or `max` have neither length 1 nor any consistent pattern. That's no coincidence — unless the divisor is `‖v‖₂`, there's no way to guarantee the result's length equals 1.

### A small proof: why the length of `v / c` is `‖v‖₂ / c` ("homogeneity" of the norm)

The above was a numerical check; here's the general derivation, proving that "dividing by its own L2 norm always gives a norm of 1" holds for **any** vector, with no need to plug in specific numbers each time. Let `v = [v₁, v₂, ..., vₙ]` and let `c > 0` be any positive number:

```
‖v/c‖₂ = √( (v₁/c)² + (v₂/c)² + ... + (vₙ/c)² )
       = √( (1/c²) · (v₁² + v₂² + ... + vₙ²) )
       = √(1/c²) · √(v₁² + v₂² + ... + vₙ²)          ← a square root splits into a product of two square roots
       = (1/c) · ‖v‖₂                                 ← because √(1/c²) = 1/c when c>0
```

This property is called "homogeneity": scale the whole vector by `1/c` and its norm scales by `1/c` too. Now set the `c` in this general result to `‖v‖₂` itself:

```
‖ v / ‖v‖₂ ‖₂  =  ‖v‖₂ / ‖v‖₂  =  1
```

That's a rigorous proof: as long as you divide by the vector's own L2 norm, the result's L2 norm is always exactly 1 — a necessary consequence of the formula, not something that "happens to work for the example [3,4]."

### Three things to pin down before you write `normalize`

- Input: `v`, a float vector of shape `(n,)`
- Key step: compute `length = np.linalg.norm(v)`, then do `v / length`
- Return: a unit vector with the same shape as `v` and an L2 norm of 1

In [ ]:
def normalize(v):
    # ✏️ TODO: return the unit-length vector
    # Hint: length = np.linalg.norm(v); if length == 0 return v (all zeros), otherwise return v / length
    raise NotImplementedError("TODO: divide v by its L2 norm")

In [ ]:
try:
    u = normalize(np.array([3.0, 4.0]))
    print("normalized:", u, "| length:", round(float(np.linalg.norm(u)), 6))
    assert abs(np.linalg.norm(u) - 1.0) < 1e-9, "after normalizing, the length should be 1"
    assert np.allclose(u, [0.6, 0.8], atol=1e-9), "wrong direction: v=[3,4] should normalize to [0.6, 0.8]"

    # Negative components: the direction isn't affected by the sign
    u_neg = normalize(np.array([-6.0, 8.0]))
    assert abs(np.linalg.norm(u_neg) - 1.0) < 1e-9, "a vector with a negative component should still have length 1 after normalizing"
    assert np.allclose(u_neg, [-0.6, 0.8], atol=1e-9), "wrong direction: v=[-6,8] should normalize to [-0.6, 0.8]"

    # Zero vector: guarded, so no nan and no exception
    z = normalize(np.array([0.0, 0.0]))
    assert not np.any(np.isnan(z)), "normalizing the zero vector should not produce nan"
    print("\n✅ Passed: the [3,4] direction, the negative-component direction and the zero vector — all three edge cases.")
except (NotImplementedError, TypeError) as e:
    print(f"⚠️  Not implemented yet — please complete normalize(): {e}")


**🔗 Aurora connection**: `normalize(v)` is called on each frame's features in Aurora's MFCC pipeline; after normalization, cosine similarity is the same as `np.dot(a, b)`. Speaker recognition compares the direction of d-vectors, not their magnitude — two recordings can differ a lot in volume, but if their directions are close after normalization, they're judged to be the same speaker.

### Check: why does "dot product after normalization = cosine similarity"?

Recall last lesson's formula: `cos(θ) = (a·b) / (‖a‖·‖b‖)`. If `a` and `b` are first normalized to unit vectors `â = a/‖a‖`, `b̂ = b/‖b‖` (so `‖â‖=‖b̂‖=1`), then:

```
â · b̂ = (a/‖a‖) · (b/‖b‖) = (a·b) / (‖a‖·‖b‖) = cos(θ)
```

In other words, `â · b̂` and the original `cos(θ)` formula are **the same expression** — the division has just been done up front, on the vectors. It's not a coincidence; it's an algebraic identity. Below we check it with two pairs of vectors: the first pair, `a=[3,4]` and `b=[6,8]`, has "the same direction but different magnitudes" (in Aurora, two recordings with the same timbre but different volume); the second pair points in different directions, to confirm the identity doesn't only hold when the directions match.

In [ ]:
# Pair 1: same direction, different magnitudes (b is exactly a scaled by 2)
a_raw = np.array([3.0, 4.0])
b_raw = np.array([6.0, 8.0])
cos_raw = np.dot(a_raw, b_raw) / (np.linalg.norm(a_raw) * np.linalg.norm(b_raw))
a_unit = a_raw / np.linalg.norm(a_raw)
b_unit = b_raw / np.linalg.norm(b_raw)
dot_unit = np.dot(a_unit, b_unit)
print('Pair 1: a=[3,4], b=[6,8]')
print('  original cosine similarity cos(θ) =', cos_raw)
print('  dot product after normalizing â·b̂ =', dot_unit)
print('  equal:', np.isclose(cos_raw, dot_unit))

# Pair 2: different directions, to confirm it isn't a coincidence that only holds for matching directions
a2, b2 = np.array([1.0, 0.0]), np.array([1.0, 1.0])
cos2 = np.dot(a2, b2) / (np.linalg.norm(a2) * np.linalg.norm(b2))
dot2 = np.dot(a2 / np.linalg.norm(a2), b2 / np.linalg.norm(b2))
print('\nPair 2: a=[1,0], b=[1,1]')
print('  original cosine similarity cos(θ) =', cos2)
print('  dot product after normalizing â·b̂ =', dot2)
print('  equal:', np.isclose(cos2, dot2))


## 🎨 Picture: L2 norm = arrow length, (3,4) → 5

In [ ]:
from laviz import style, arrows2d
style()
arrows2d([[3,4]], ['v, |v|=5'], title='norm = length');

## Behind the curtain: L1, L2 and L∞ are three special cases of one formula

So far, L1, L2 and L∞ look like three unrelated formulas you have to memorize separately. In fact they all come out of the same "mold" — one general formula with different values of `p` plugged in:

```
‖v‖ₚ = ( |v₁|^p + |v₂|^p + ... + |vₙ|^p )^(1/p)
```

Plug in a few specific values of `p` and the familiar formulas pop out on their own:

- **p=1**: `‖v‖₁ = (|v₁|¹+...+|vₙ|¹)^(1/1) = |v₁|+...+|vₙ|` — that's L1 (sum of absolute values).
- **p=2**: `‖v‖₂ = (|v₁|²+...+|vₙ|²)^(1/2) = √(v₁²+...+vₙ²)` — that's L2 (Pythagoras).
- **p→∞**: as `p` gets very large, the biggest `|vᵢ|^p` in the formula leaves all the other terms far behind (for example, `5¹⁰⁰` is many orders of magnitude bigger than `3¹⁰⁰`, so all the other terms added together can be ignored). The sum is then decided almost entirely by the largest component, and after taking the `p`-th root it approaches `max(|vᵢ|)` — that's L∞.

That's also why the `p` argument of `np.linalg.norm(v, p)` can take `1`, `2` or `np.inf` to give the three norms — they were always three special cases of one formula, not three separate functions.

So what's the "unit ball" good for (the set of all vectors with `‖v‖ₚ=1`, whose L1/L2/L∞ shapes we just drew)? The answer is direct: what the `normalize` function does is exactly "take any vector and scale it along its own direction until it lands precisely on the surface of the unit ball" — the shape of the unit ball is a map of all the vectors whose "length is exactly 1." The code below checks what relationship the coordinates of a point on the unit-ball boundary must satisfy for different values of `p`.

In [ ]:
import numpy as np

# For different p: given x1, find x2 so that the point (x1, x2) lies on the Lp unit ball (in the first quadrant)
# Lp ball: |x1|^p + |x2|^p = 1 => x2 = (1 - |x1|^p)^{1/p}
x1 = 0.6
for p in [1, 2, 4, np.inf]:
    if np.isinf(p):
        x2 = 1.0   # the L∞ ball is a square; the point (0.6, 1.0) lies on its top edge (the actual corners are (±1, ±1))
    else:
        val = max(0.0, 1 - abs(x1)**p)
        x2 = val**(1/p)
    lbl = '∞' if np.isinf(p) else str(int(p))
    print(f'on the L{lbl} ball with x1={x1}: x2={x2:.4f}  (norm check={np.linalg.norm([x1,x2],p if not np.isinf(p) else np.inf):.4f})')


## Parameter experiment: turn one knob at a time

Scale `v=[3, 4]` up 10 times to `[30, 40]` and compare the normalized result — it should still be `[0.6, 0.8]`: the direction doesn't change with magnitude. Then try a bare `v / np.linalg.norm(v)` on the all-zero vector `[0, 0]` and watch it produce `nan` (NumPy also prints a `RuntimeWarning`) — that's why real code has to catch the zero vector first: `if np.linalg.norm(v) == 0: return v`, and only then send non-zero vectors through `v / np.linalg.norm(v)`.

### Laying the groundwork: before L1/L2 regularization, get clear on what a "gradient" is

Picture the loss function as a mountain you're standing on, where the "altitude" is "how wrong the model is." Your goal is to walk down to the lowest point of the valley (minimum loss). The **gradient** is "the direction, from where you're standing right now, in which the altitude rises fastest" — it has both a direction and a size, like an arrow.

When training a model we use it the other way round: take a small step in the **opposite** direction of the gradient, and the loss drops a tiny bit. That's "gradient descent," and as a formula it reads:

```
w_new = w - lr * gradient
```

- `lr` (the learning rate) controls how big each step is;
- the **size** of the gradient tells us "how hard to push on this step" — a large gradient means the loss changes sharply in that direction, so the step goes further; a small gradient means you barely move.

The rigorous mathematical definition is "the derivative of the loss function with respect to the parameters `w`" (for a function of several variables, this derivative is called the gradient). If you haven't learned derivatives yet, don't worry — below, just take the L1 and L2 gradient formulas as "experimental facts." That won't get in the way of the core conclusion, "L1 makes weights sparse, L2 only shrinks them proportionally"; if you'd like to see the derivation, there's an optional one right after the formulas.

## 4. What regularization means: how the L1 and L2 gradients differ

L1 regularization (Lasso) adds `λ‖w‖₁` to the loss function, and its gradient with respect to the parameters `w` is `λ·sign(w)` — however small `w` gets, the gradient's absolute value is always λ, which pushes small weights all the way to zero (**sparsity**).

L2 regularization (Ridge / weight decay) adds `λ‖w‖₂²`, and its gradient is `2λw` — the smaller the weight, the smaller the gradient, so it only shrinks weights proportionally and never makes them exactly zero (**shrinkage**).

Intuitive contrast: L1 applies a "pull" of fixed size to every weight; L2 applies a "spring force" proportional to the weight's size.


### Derivation (optional, for the curious): how do you get the gradients of `λ‖w‖₁` and `λ‖w‖₂²`?

If you don't want to see the differentiation just now, skip this part and simply use the two conclusions above as "experimental facts" — the next code cell checks them numerically.

**The L2 term**: `λ‖w‖₂² = λ·(w₁² + w₂² + ... + wₙ²)`. Pick out just one component `wᵢ`, treat it as the variable, and differentiate (treating the other components as constants):

```
d/dwᵢ [ λ·wᵢ² ] = λ · 2wᵢ = 2λwᵢ
```

All this uses is the most basic power rule: `d/dx(x²) = 2x`. So the L2 gradient is `2λw` — the smaller the weight `w`, the smaller the gradient, linearly.

**The L1 term**: `λ‖w‖₁ = λ·(|w₁| + |w₂| + ... + |wₙ|)`. The derivative of the absolute-value function `|wᵢ|` is:

```
d/dwᵢ |wᵢ| =  +1   if wᵢ > 0
             -1   if wᵢ < 0
             undefined (sharp corner)  if wᵢ = 0
```

Multiply by the coefficient `λ` and you get `λ·sign(wᵢ)`. **The key point is at `wᵢ=0`**: the graph of the absolute-value function has a sharp corner at 0 (it doesn't round off smoothly the way `x²` does), so mathematically there's no single tangent slope, and strictly speaking the "derivative" isn't defined at that point. The engineering fix is simply to agree that `sign(0)=0` (that is, once `w` has reached 0, stop pushing it), so the gradient formula works over the whole domain. A "derivative" that has been given a sensible agreed value at a sharp corner like this is mathematically called a **subgradient** — you don't need to remember the name, just know that it's a patch so the formula also works at the corner.

In [ ]:
# Numerical simulation: gradient descent using only the L1 regularization gradient — watch w close in on 0 step by step
w, lr, lam = 0.3, 0.1, 0.1
print(f'start: w={w:.4f}  (lr={lr}, λ={lam})\n')
for step in range(1, 11):
    grad = lam * np.sign(w) if w != 0 else 0.0   # convention sign(0)=0: once at 0, stop pushing
    w_next = w - lr * grad
    if w != 0 and np.sign(w_next) != np.sign(w):  # if a step overshoots 0, stop at 0 rather than going negative
        w_next = 0.0
    w = w_next
    print(f'step {step:>2}: grad={grad:+.4f}   w={w:.4f}')

print(f"\nEach step subtracts a fixed lr·λ = {lr*lam} (because the L1 gradient is always ±λ, whatever the size of w).")
print(f"Going from 0.3 to exactly 0 takes 0.3 / {lr*lam} = {0.3/(lr*lam):.0f} steps;")
print("after that sign(0)=0 (by convention), the gradient is zero, and w stays exactly at 0 without going negative — that's why L1 makes weights exactly sparse.")
print("Contrast: the L2 gradient is 2λw, so the update is w ← w·(1-2λ·lr) — it just keeps discounting, getting ever closer to 0 in theory but never exactly 0.")


In [ ]:
import numpy as np

# Comparing the L1 and L2 regularization gradients (the direction of the weight update in gradient descent)
weights = np.array([2.0, 0.5, 0.05, 0.001])   # four weights of different sizes
lambda_ = 0.1

print("weight w    L1 grad λ·sign(w)    L2 grad 2λ·w")
print('-' * 52)
for w in weights:
    grad_l1 = lambda_ * np.sign(w)     # always ±λ, regardless of the size of w
    grad_l2 = 2 * lambda_ * w          # proportional to w
    print(f"  w={w:.3f}       grad_L1={grad_l1:+.4f}         grad_L2={grad_l2:+.6f}")

print()
print("Conclusion:")
print("  L1 gradient is always ±λ: however small the weight, it's pushed toward zero just as hard → sparse (many weights exactly zero)")
print("  L2 gradient is proportional to w: small weights feel a weaker push              → shrinks only, never zero (weight decay)")


## Wrap-up

You can now use `normalize(v)` to put any non-zero vector onto the unit ball, with its L2 length computed by `np.linalg.norm(v)`. Aurora's MFCC pipeline calls `normalize` on each frame's features before they go into the classifier, so that energy values from different recording conditions sit on the same scale. The MSE loss `mean((y_pred - y_true)**2)` is the same as taking the L2 norm of the error vector, squaring it and averaging. Next lesson: in **L12**'s matrix multiplication, the L2 norm of each column of a matrix determines how much it stretches the input along that column's direction.

(A tiny sneak preview of the intuition: matrix multiplication `Av` takes each component of `v`, spreads it out along the direction of the matching "column" of the matrix, and adds the results up. If one column has a large L2 norm, the input's contribution along that column gets amplified more. So "the length of that column" directly determines how much the matrix stretches the input in that direction — just keep the impression for now; L12 goes into it in detail, with pictures.)

### Taking it apart: why "MSE loss" equals "the squared L2 norm of the error vector"

Put the two formulas side by side:

```
MSE     = mean( (y_pred - y_true)² )   = (1/n) · Σᵢ (y_pred,ᵢ - y_true,ᵢ)²
‖e‖₂²   = ‖y_pred - y_true‖₂²          =        Σᵢ (y_pred,ᵢ - y_true,ᵢ)²
```

Here `e = y_pred - y_true` is the "error vector" (each sample's prediction minus its true value, collected into one vector). Comparing the two lines: **the sums are identical** — both "square each error component and add them up"; the only difference is that MSE also divides by `n` (the number of samples), i.e. it takes one extra average. So, strictly:

```
MSE = ‖e‖₂² / n
```

MSE is nothing more and nothing less than "the squared L2 norm of the error vector," divided by the number of samples so that the loss doesn't grow linearly as you add more samples. The code below checks this identity:

In [ ]:
y_pred = np.array([2.5, 0.0, 2.1, 7.8])
y_true = np.array([3.0, -0.5, 2.0, 7.0])
e = y_pred - y_true

mse = np.mean(e ** 2)
l2_sq_over_n = np.linalg.norm(e) ** 2 / len(e)
print('MSE              =', mse)
print('‖e‖₂² / n        =', l2_sq_over_n)
print('equal:', np.isclose(mse, l2_sq_over_n))


---
⬇️ **Completion check**: you've read the wrap-up; please finish the whiteboard challenge below before ticking the self-review.


## ✏️ Whiteboard challenge: norms by hand (target: 8 minutes)

Cover the screen and answer on paper:

**Vectors**: v = [3, 4], u = [1, 0, 0], w = [-1, 2, -3]

**Question 1**: What is the L2 norm of v? Its L1 norm? Its L∞ norm?

**Question 2**: What is normalize(v)? Check that its L2 norm = 1.
Formula: normalize(v) = v / ‖v‖₂

**Question 3**: What is the L2 distance between the two vectors a = [1, 2] and b = [4, 6]?
Formula: ‖a - b‖₂ = √((4-1)² + (6-2)²) = ?

**Question 4**: What is the L∞ norm of the vector w = [-1, 2, -3]?

When you've finished, run the cell below to check your answers.

In [ ]:
# ✏️ Answer-check cell
import numpy as np

v = np.array([3.0, 4.0])

# Q1: the three norms
assert np.isclose(np.linalg.norm(v, 2), 5.0, atol=1e-10)
assert np.isclose(np.linalg.norm(v, 1), 7.0, atol=1e-10)
assert np.isclose(np.linalg.norm(v, np.inf), 4.0, atol=1e-10)
print(f"Q1 ✅  v=[3,4]: L2={np.linalg.norm(v):.1f}, L1={np.linalg.norm(v,1):.1f}, L∞={np.linalg.norm(v,np.inf):.1f}")

# Q2: normalization
try:
    nv = normalize(v)
    assert np.isclose(np.linalg.norm(nv), 1.0, atol=1e-9)
    print(f"Q2 ✅  normalize([3,4]) = {np.round(nv,4)}  ‖‖₂ = {np.linalg.norm(nv):.6f}")
except (NotImplementedError, TypeError):
    print("⬜ Q2: implement normalize() first, then run the answer-check cell")

# Q3: L2 distance
a, b = np.array([1.0, 2.0]), np.array([4.0, 6.0])
dist = np.linalg.norm(a - b)
assert np.isclose(dist, 5.0, atol=1e-10)  # √(9+16) = 5
print(f"Q3 ✅  ‖[1,2]-[4,6]‖₂ = √(9+16) = {dist:.1f}")

# Q4: L∞ norm
w = np.array([-1.0, 2.0, -3.0])
linf = np.linalg.norm(w, np.inf)
assert np.isclose(linf, 3.0, atol=1e-10)
print(f"Q4 ✅  ‖[-1,2,-3]‖∞ = max(1,2,3) = {linf:.1f}")
print("\n🎉 Norms whiteboard challenge passed! The three 'rulers' are now second nature.")

In [ ]:
# ✏️ Self-review for this lesson
l11_review = {
    "normalize_implemented":  None,  # normalize implemented and passes its asserts? True/False
    "three_norms_understood": None,  # can state the formulas and geometric meaning of L1/L2/L∞? True/False
    "regularization_intuition": None, # understand L1 → sparse, L2 → weight decay? True/False
    "whiteboard_passed":      None,  # finished the whiteboard challenge on paper? True/False
}

unfilled = [k for k, v in l11_review.items() if v is None]
assert not unfilled, f'Not filled in yet: {unfilled}'
weak = [k for k, v in l11_review.items() if v is False]
if weak:
    print(f'⚠️  Needs more work: {weak}')
else:
    print('✅ L11 complete! On to L12: matrix multiplication')

---

→ **Next lesson** · [L12 · Matrix multiplication](L12_matrices.ipynb)

> Next lesson covers **matrix multiplication**: a matrix is a linear map, multiplication is composition, and a 2×2 example worked by hand.